# Resume Screening with LLMs

**Goal**: Build a resume scoring system using structured outputs and submit scores to the leaderboard.

## Setup

In [ ]:
import json
from pydantic import BaseModel, Field
from typing import List
from resume_utils import load_resumes, load_job_requirements, analyze_resume, submit_score, delete_score, delete_team

# --- Configuration ---
OPENROUTER_API_KEY = ""  # Paste your key here
MODEL = "anthropic/claude-sonnet-4-6"
TEAM_NAME = ""  # Set your team name
LEADERBOARD_URL = "http://ai-leaderboard.site"

if not OPENROUTER_API_KEY.strip():
    raise RuntimeError(
        "Please set OPENROUTER_API_KEY above before running this notebook.\n"
        "Get your key from: https://openrouter.ai/keys"
    )

if not TEAM_NAME.strip():
    raise RuntimeError(
        "Please set your TEAM_NAME above before running this notebook."
    )

# Load resume data
resumes = load_resumes("../data/resumes_final.csv")

print(f"API key configured")
print(f"Model: {MODEL}")
print(f"Team: {TEAM_NAME}")
print(f"Loaded {len(resumes)} resumes")

## Example: Score a Resume and Submit to Leaderboard

In [ ]:
# Grab a single resume
sample_id = list(resumes.keys())[0]
sample_resume = resumes[sample_id]

# Define the output structure
class ResumeScore(BaseModel):
    score: int
    reasoning: str

# Score a resume using structured output
prompt = "On a scale of 1 to 100, how good of a fit is this person for a software engineering job?"

result = analyze_resume(
    OPENROUTER_API_KEY,
    prompt,
    sample_resume["Resume_str"],
    ResumeScore,
    model=MODEL,
)

if result["error"]:
    print(f"Error: {result['error']}")
else:
    score = result["result"]["score"]
    reasoning = result["result"]["reasoning"]
    print(f"Resume {sample_id} score: {score}/100")
    print(f"Reasoning: {reasoning}")
    print(f"Tokens used: {result['usage'].get('total_tokens', 0)}")

    # Submit to leaderboard
    response = submit_score(TEAM_NAME, sample_id, score, api_url=LEADERBOARD_URL)
    print(f"\nSubmitted to leaderboard: {response}")

## Multi-Step Resume Scoring

Instead of asking the LLM to score in one shot, we break it into steps:

1. **Extract** the top 3 most relevant skills and years of experience for each
2. **Extract** education level and notable projects/achievements
3. **Combine** the extractions into a final scoring prompt

This mirrors the "Crawl → Walk → Run" methodology from the slides — we define clear inputs and outputs for each component.

In [ ]:
# Load the job requirements
job_req = load_job_requirements("../data/job_req_senior.md")
print(f"Job req loaded ({len(job_req)} chars)")
print(job_req[:300] + "...")

# --- Step 1 schema: Extract relevant skills + years of experience ---
class SkillMatch(BaseModel):
    skill: str = Field(description="A skill from the resume that is relevant to the job requirements")
    years_experience: float = Field(description="Estimated years of experience with this skill based on the resume")
    evidence: str = Field(description="Brief quote or reference from the resume supporting this")

class SkillsExtraction(BaseModel):
    top_skills: List[SkillMatch] = Field(description="The top 3 skills most relevant to the job requirements")
    total_years_experience: float = Field(description="Total years of professional experience mentioned")

# --- Step 2 schema: Extract education and achievements ---
class EducationExtraction(BaseModel):
    highest_degree: str = Field(description="Highest degree obtained (e.g., 'Bachelor's in CS', 'No degree mentioned')")
    relevant_certifications: List[str] = Field(description="Any certifications relevant to the job")
    notable_achievements: List[str] = Field(description="Up to 3 notable projects or achievements relevant to the role")

# --- Step 3 schema: Final score ---
class FinalScore(BaseModel):
    score: int = Field(description="Overall fit score from 0-100")
    strengths: List[str] = Field(description="Top 2-3 strengths for this role")
    weaknesses: List[str] = Field(description="Top 2-3 gaps or weaknesses for this role")
    reasoning: str = Field(description="Brief overall assessment")


def score_resume(resume_text: str, job_req: str, api_key: str, model: str) -> dict:
    """Score a resume in multiple steps: extract info, then combine for final score."""

    # Step 1: Extract skills and experience
    skills_result = analyze_resume(
        api_key,
        f"""Given the following job requirements, extract the top 3 most relevant skills
from the resume and estimate years of experience for each.

Job Requirements:
{job_req}""",
        resume_text,
        SkillsExtraction,
        model=model,
    )

    if skills_result["error"]:
        return {"error": f"Step 1 (skills) failed: {skills_result['error']}", "score": None}

    # Step 2: Extract education and achievements
    edu_result = analyze_resume(
        api_key,
        f"""Given the following job requirements, extract the candidate's education,
relevant certifications, and up to 3 notable achievements or projects.

Job Requirements:
{job_req}""",
        resume_text,
        EducationExtraction,
        model=model,
    )

    if edu_result["error"]:
        return {"error": f"Step 2 (education) failed: {edu_result['error']}", "score": None}

    # Step 3: Combine extractions into a final score
    skills_summary = json.dumps(skills_result["result"], indent=2)
    edu_summary = json.dumps(edu_result["result"], indent=2)

    scoring_prompt = f"""You are scoring a resume for fit against a job posting.
Based on the extracted information below, provide a score from 0-100.

Job Requirements:
{job_req}

Extracted Skills & Experience:
{skills_summary}

Extracted Education & Achievements:
{edu_summary}

Score the candidate's overall fit. Be calibrated:
- 80-100: Strong match, meets most required qualifications
- 60-79: Decent match, meets some requirements but has gaps
- 40-59: Weak match, significant gaps in required skills
- 0-39: Poor match, missing most required qualifications"""

    final_result = analyze_resume(
        api_key,
        scoring_prompt,
        "",  # resume already summarized in the prompt
        FinalScore,
        model=model,
    )

    if final_result["error"]:
        return {"error": f"Step 3 (scoring) failed: {final_result['error']}", "score": None}

    # Combine usage across all steps
    total_tokens = sum(
        r.get("usage", {}).get("total_tokens", 0)
        for r in [skills_result, edu_result, final_result]
    )

    return {
        "error": None,
        "score": final_result["result"]["score"],
        "skills": skills_result["result"],
        "education": edu_result["result"],
        "final": final_result["result"],
        "total_tokens": total_tokens,
    }

In [ ]:
# --- Test on a single resume ---
result = score_resume(sample_resume["Resume_str"], job_req, OPENROUTER_API_KEY, MODEL)

if result["error"]:
    print(f"Error: {result['error']}")
else:
    print(f"=== Resume {sample_id} ===")
    print(f"\nStep 1 - Skills:")
    for s in result["skills"]["top_skills"]:
        print(f"  {s['skill']}: {s['years_experience']} yrs — {s['evidence']}")
    print(f"  Total experience: {result['skills']['total_years_experience']} yrs")

    print(f"\nStep 2 - Education:")
    print(f"  Degree: {result['education']['highest_degree']}")
    print(f"  Certs: {result['education']['relevant_certifications']}")
    print(f"  Achievements: {result['education']['notable_achievements']}")

    print(f"\nStep 3 - Final Score: {result['score']}/100")
    print(f"  Strengths: {result['final']['strengths']}")
    print(f"  Weaknesses: {result['final']['weaknesses']}")
    print(f"  Reasoning: {result['final']['reasoning']}")
    print(f"\n  Total tokens used: {result['total_tokens']}")

    # Submit to leaderboard
    response = submit_score(TEAM_NAME, sample_id, result["score"], api_url=LEADERBOARD_URL)
    print(f"\nSubmitted to leaderboard: {response}")

## Walk: Run on a Sample of Resumes

Now that the vertical slice works for one resume, let's expand to a small batch.

In [ ]:
# Score a sample of 5 resumes
sample_ids = list(resumes.keys())[:5]
results = {}
total_tokens = 0

for rid in sample_ids:
    print(f"Scoring resume {rid}...", end=" ")
    r = score_resume(resumes[rid]["Resume_str"], job_req, OPENROUTER_API_KEY, MODEL)

    if r["error"]:
        print(f"ERROR: {r['error']}")
        continue

    results[rid] = r
    total_tokens += r["total_tokens"]
    print(f"Score: {r['score']}/100  ({r['total_tokens']} tokens)")
    response = submit_score(TEAM_NAME, rid, r['score'], api_url=LEADERBOARD_URL)
    print(f"Submitted to leaderboard: {response}")

# Summary
print(f"\n{'='*50}")
print(f"Scored {len(results)}/{len(sample_ids)} resumes")
print(f"Total tokens used: {total_tokens}")
print(f"\nResults:")
for rid, r in sorted(results.items(), key=lambda x: x[1]["score"], reverse=True):
    skills = ", ".join(s["skill"] for s in r["skills"]["top_skills"])
    print(f"  {rid}: {r['score']}/100 — Top skills: {skills}")

## Clean Up: Remove the Example Submission

In [ ]:
# Remove the single example submission we just made
response = delete_score(TEAM_NAME, sample_id, api_url=LEADERBOARD_URL)
print(f"Deleted: {response}")

# To delete ALL submissions for your team, uncomment:
# response = delete_team(TEAM_NAME, api_url=LEADERBOARD_URL)
# print(f"Deleted all: {response}")